## Import essential libraries and Configure settings

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sb
sb.set()

## Import and Inspect data

The CSV contains 70,000 records and 13 columns. The data types below are the pandas dtypes observed in this file. `id` is a record identifier, not one of Kaggle's 11 predictor features.

| Variable | Data type | Feature group | Variable type | Description |
|---|---|---|---|---|
| `id` | `int64` | Identifier | Categorical (identifier) | Unique record identifier; not a clinical measurement or model feature. |
| `age` | `int64` | Objective | Numerical | Age at the time of examination, recorded in days. |
| `gender` | `int64` | Objective | Categorical (nominal) | Categorical code for gender (observed codes: 1 and 2; the dataset description does not specify their label mapping). |
| `height` | `int64` | Objective | Numerical | Height in centimetres. |
| `weight` | `float64` | Objective | Numerical | Weight in kilograms. |
| `ap_hi` | `int64` | Examination | Numerical | Systolic blood pressure, in mmHg. |
| `ap_lo` | `int64` | Examination | Numerical | Diastolic blood pressure, in mmHg. |
| `cholesterol` | `int64` | Examination | Categorical (ordinal) | Cholesterol level: 1 = normal, 2 = above normal, 3 = well above normal. |
| `gluc` | `int64` | Examination | Categorical (ordinal) | Glucose level: 1 = normal, 2 = above normal, 3 = well above normal. |
| `smoke` | `int64` | Subjective | Categorical (binary) | Smoking indicator: 0 = no, 1 = yes. |
| `alco` | `int64` | Subjective | Categorical (binary) | Alcohol-intake indicator: 0 = no, 1 = yes. |
| `active` | `int64` | Subjective | Categorical (binary) | Physical-activity indicator: 0 = no, 1 = yes. |
| `cardio` | `int64` | Target | Categorical (binary) | Cardiovascular disease: 0 = absent, 1 = present. |

In [3]:
data = pd.read_csv("../data/cardio_train.csv", sep = ';')
data.head()

,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
0,0,18393,2,168,62.0,110,80,1,1,0,0,1,0
1,1,20228,1,156,85.0,140,90,3,1,0,0,1,1
2,2,18857,1,165,64.0,130,70,3,1,0,0,0,1
3,3,17623,2,169,82.0,150,100,1,1,0,0,1,1
4,4,17474,1,156,56.0,100,60,1,1,0,0,0,0


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70000 entries, 0 to 69999
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id           70000 non-null  int64  
 1   age          70000 non-null  int64  
 2   gender       70000 non-null  int64  
 3   height       70000 non-null  int64  
 4   weight       70000 non-null  float64
 5   ap_hi        70000 non-null  int64  
 6   ap_lo        70000 non-null  int64  
 7   cholesterol  70000 non-null  int64  
 8   gluc         70000 non-null  int64  
 9   smoke        70000 non-null  int64  
 10  alco         70000 non-null  int64  
 11  active       70000 non-null  int64  
 12  cardio       70000 non-null  int64  
dtypes: float64(1), int64(12)
memory usage: 6.9 MB


In [5]:
data["age_years"] = data["age"] / 365.25

In [6]:
data.describe()

,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio,age_years
count,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000
mean,49972.419900,19468.865814,1.349571,164.359229,74.205690,128.817286,96.630414,1.366871,1.226457,0.088129,0.053771,0.803729,0.499700,53.302850
std,28851.302323,2467.251667,0.476838,8.210126,14.395757,154.011419,188.472530,0.680250,0.572270,0.283484,0.225568,0.397179,0.500003,6.754967
min,0.000000,10798.000000,1.000000,55.000000,10.000000,-150.000000,-70.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,29.563313
25%,25006.750000,17664.000000,1.000000,159.000000,65.000000,120.000000,80.000000,1.000000,1.000000,0.000000,0.000000,1.000000,0.000000,48.361396
50%,50001.500000,19703.000000,1.000000,165.000000,72.000000,120.000000,80.000000,1.000000,1.000000,0.000000,0.000000,1.000000,0.000000,53.943874
75%,74889.250000,21327.000000,2.000000,170.000000,82.000000,140.000000,90.000000,2.000000,1.000000,0.000000,0.000000,1.000000,1.000000,58.390144
max,99999.000000,23713.000000,2.000000,250.000000,200.000000,16020.000000,11000.000000,3.000000,3.000000,1.000000,1.000000,1.000000,1.000000,64.922656


## Data quality and variable distributions

These checks report missing values, duplicate rows/IDs, category frequencies, and blood-pressure values that may need review. Outlier flags are prompts for investigation, not automatic exclusion rules; the source data is left unchanged.

In [ ]:
from pathlib import Path
from IPython.display import display

plot_dir = Path("plot")
plot_dir.mkdir(parents=True, exist_ok=True)

numerical_variables = ["age_years", "height", "weight", "ap_hi", "ap_lo"]
categorical_variables = ["gender", "cholesterol", "gluc", "smoke", "alco", "active"]

print(f"Rows: {len(data):,}; columns: {data.shape[1]}")
print(f"Duplicate rows: {data.duplicated().sum():,}")
print(f"Duplicate IDs: {data['id'].duplicated().sum():,}")
display(pd.DataFrame({
    "dtype": data.dtypes.astype(str),
    "missing_values": data.isna().sum(),
}))

bp_checks = pd.DataFrame({
    "check": [
        "Rows with non-positive systolic or diastolic pressure",
        "Rows where diastolic pressure is at least systolic pressure",
    ],
    "rows_flagged": [
        ((data["ap_hi"] <= 0) | (data["ap_lo"] <= 0)).sum(),
        (data["ap_lo"] >= data["ap_hi"]).sum(),
    ],
})
display(bp_checks)

bp_outlier_rows = []
for variable in ["ap_hi", "ap_lo"]:
    q1, q3 = data[variable].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr
    outside_fences = (data[variable] < lower_fence) | (data[variable] > upper_fence)
    bp_outlier_rows.append({
        "variable": variable,
        "lower_IQR_fence": lower_fence,
        "upper_IQR_fence": upper_fence,
        "values_outside_fences": outside_fences.sum(),
    })
display(pd.DataFrame(bp_outlier_rows).round(2))

print("Numerical-variable summary:")
display(data[numerical_variables].describe().T.round(2))

category_counts = pd.concat(
    [
        data[variable].value_counts(dropna=False).rename_axis("code").reset_index(name="count").assign(variable=variable)
        for variable in categorical_variables + ["cardio"]
    ],
    ignore_index=True,
)[["variable", "code", "count"]]
print("Categorical feature and target counts:")
display(category_counts)

target_summary = data["cardio"].value_counts().sort_index().rename_axis("cardio").to_frame("count")
target_summary["percent"] = 100 * target_summary["count"] / len(data)
display(target_summary.round(2))

### Overall feature distributions

The numerical histograms limit the horizontal axis to the 0.5th–99.5th percentiles so extreme values do not hide the main distribution; observations outside the displayed range are not removed from `data`. Categorical bars show the observed integer codes. In particular, keep gender labels as codes unless a verified code mapping is available.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, variable in zip(axes.flat, numerical_variables):
    sb.histplot(data=data, x=variable, bins=30, ax=ax)
    lower, upper = data[variable].quantile([0.005, 0.995])
    ax.set_xlim(lower, upper)
    ax.set_title(f"{variable} distribution")
    ax.set_xlabel(variable.replace("_", " ").title())
sb.countplot(data=data, x="cardio", order=[0, 1], ax=axes.flat[5])
axes.flat[5].set_title("Target distribution")
axes.flat[5].set_xlabel("Cardio (0 = no, 1 = yes)")
fig.tight_layout()
fig.savefig(plot_dir / "numerical_and_target_distributions.png", dpi=150)
plt.close(fig)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, variable in zip(axes.flat, categorical_variables):
    sb.countplot(data=data, x=variable, order=sorted(data[variable].dropna().unique()), ax=ax)
    ax.set_title(f"{variable} category counts")
    ax.set_xlabel(variable.replace("_", " ").title() + " (coded value)")
    ax.set_ylabel("Number of records")
fig.tight_layout()
fig.savefig(plot_dir / "categorical_feature_counts.png", dpi=150)
plt.close(fig)
print("Saved overall numerical/target and categorical distribution plots.")

## Plotting `cardio` against other variables

The plots below compare numerical variables between the two `cardio` groups using horizontal boxplots, show outcome proportions for categorical variables, and visualize Spearman correlations for continuous measurements and the binary target. Each plot is saved as a PNG in `plot/` relative to the notebook's working directory. The record identifier (`id`) and integer-coded categories are excluded from the correlation calculation.

In [ ]:
from pathlib import Path

plot_dir = Path("plot")
plot_dir.mkdir(parents=True, exist_ok=True)

numerical_variables = ["age_years", "height", "weight", "ap_hi", "ap_lo"]
categorical_variables = ["gender", "cholesterol", "gluc", "smoke", "alco", "active"]
numerical_labels = {
    "age_years": "Age (years)",
    "height": "Height (cm)",
    "weight": "Weight (kg)",
    "ap_hi": "Systolic blood pressure (mmHg)",
    "ap_lo": "Diastolic blood pressure (mmHg)",
}

for variable in numerical_variables:
    plot_data = pd.DataFrame({
        "cardio": data["cardio"],
        "value": data[variable],
    })
    fig, ax = plt.subplots(figsize=(8, 5))
    sb.boxplot(
        data=plot_data,
        x="value",
        y="cardio",
        order=[0, 1],
        orient="h",
        showfliers=False,
        ax=ax,
    )
    ax.set_title(f"{numerical_labels[variable]} by cardiovascular disease status")
    ax.set_xlabel(numerical_labels[variable])
    ax.set_ylabel("Cardiovascular disease (0 = no, 1 = yes)")
    ax.set_yticks([0, 1], labels=["No", "Yes"])
    fig.tight_layout()
    fig.savefig(plot_dir / f"cardio_{variable}.png", dpi=150)
    plt.close(fig)

for variable in categorical_variables:
    proportions = pd.crosstab(
        data[variable], data["cardio"], normalize="index"
    ).reindex(columns=[0, 1], fill_value=0)
    proportions = proportions.rename(columns={
        0: "No cardiovascular disease",
        1: "Cardiovascular disease",
    })
    fig, ax = plt.subplots(figsize=(7, 5))
    proportions.plot(kind="bar", stacked=True, ax=ax)
    ax.set_title(f"Cardiovascular disease proportion by {variable}")
    ax.set_xlabel(variable.replace("_", " ").title())
    ax.set_ylabel("Proportion of records")
    ax.set_ylim(0, 1)
    ax.legend(title="Outcome")
    fig.tight_layout()
    fig.savefig(plot_dir / f"cardio_vs_{variable}.png", dpi=150)
    plt.close(fig)

correlation_variables = numerical_variables + ["cardio"]
correlation_matrix = data[correlation_variables].corr(method="spearman")
fig, ax = plt.subplots(figsize=(9, 7))
sb.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    square=True,
    linewidths=0.5,
    ax=ax,
)
ax.set_title("Spearman correlation: continuous variables and cardio")
fig.tight_layout()
fig.savefig(plot_dir / "correlation_heatmap.png", dpi=150)
plt.close(fig)

print(f"Saved {len(numerical_variables) + len(categorical_variables) + 1} plots to {plot_dir.resolve()}")

## Cardio prevalence by age and gender code

These are descriptive outcome prevalence comparisons, not model sensitivity/recall. The dataset documentation does not provide a verified mapping from gender codes to labels, so the groups remain identified by code.

In [ ]:
age_group_labels = ["20-29", "30-39", "40-49", "50-59", "60-69"]
subgroup_data = data.assign(
    age_group=pd.cut(
        data["age_years"],
        bins=[20, 30, 40, 50, 60, 70],
        right=False,
        labels=age_group_labels,
    )
)
age_summary = (
    subgroup_data.groupby("age_group", observed=True)
    .agg(n=("cardio", "size"), prevalence=("cardio", "mean"))
    .reset_index()
)
gender_summary = (
    data.groupby("gender")
    .agg(n=("cardio", "size"), prevalence=("cardio", "mean"))
    .reset_index()
)
gender_summary["group"] = gender_summary["gender"].map(lambda code: f"Gender code {code}")
display(age_summary.assign(prevalence=age_summary["prevalence"].map(lambda value: f"{value:.1%}")))
display(gender_summary[["group", "n", "prevalence"]].assign(
    prevalence=gender_summary["prevalence"].map(lambda value: f"{value:.1%}")
))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, summary, group_column, title in [
    (axes[0], age_summary, "age_group", "By age group"),
    (axes[1], gender_summary, "group", "By gender code"),
]:
    bars = ax.bar(summary[group_column].astype(str), summary["prevalence"])
    labels = [
        f"{rate:.1%} (n={count:,})"
        for rate, count in zip(summary["prevalence"], summary["n"])
    ]
    ax.bar_label(bars, labels=labels, padding=3, fontsize=8)
    ax.set_title(title)
    ax.set_xlabel("Group")
    ax.set_ylabel("Cardio prevalence")
    ax.set_ylim(0, 0.85)
    ax.tick_params(axis="x", rotation=20)
fig.suptitle("Observed cardiovascular disease prevalence by subgroup")
fig.tight_layout()
fig.savefig(plot_dir / "cardio_prevalence_by_age_gender.png", dpi=150)
plt.close(fig)

## EDA notes and interpretation

- The CSV has 70,000 records, no missing values, and no duplicate rows or IDs. The target is nearly balanced (35,021 records with `cardio = 0`; 34,979 with `cardio = 1`).
- Blood-pressure data needs review before it is used for a clinical-rule comparison: 29 records have a non-positive systolic or diastolic value, 1,236 have diastolic pressure greater than or equal to systolic pressure, and the raw maxima are 16,020 and 11,000 mmHg. IQR fences in the checks above flag distributional outliers, not confirmed errors. Do not silently remove or correct records; agree on a documented rule with the team and report how it affects the analysis.
- In this sample, cardio prevalence increases across the age groups with adequate sample sizes: 23.9% at ages 30-39, 38.0% at 40-49, 51.9% at 50-59, and 67.0% at 60-69. The 20-29 group contains only 3 records, so its estimate is not reliable. These are associations in this dataset, not causal effects.
- Prevalence is 49.7% for gender code 1 and 50.5% for code 2. Because the supplied description does not verify the code-to-sex mapping, do not label these as male/female until the mapping is confirmed. These prevalence rates do not answer whether model sensitivity differs by group; that requires held-out model predictions and subgroup recall in the model evaluation.
- The unadjusted categorical plots show higher cardio prevalence at higher recorded cholesterol (44.0%, 60.2%, and 76.5% for codes 1-3) and glucose (48.1%, 59.3%, and 62.2% for codes 1-3). Interpret these descriptively, not as causal or adjusted effects.
- The heatmap uses Spearman correlations for continuous measurements and the binary target, leaving out `id` and integer-coded categories. Blood-pressure anomalies can still influence its results, so interpret those correlations cautiously. Correlation does not establish causation.
- Agree with the ML team on the blood-pressure rule and its clinical source/threshold before comparing it with a model. Keep the final comparison and age-/sex-group sensitivity analysis on the agreed evaluation data; do not choose a rule from test-set results.